#### If you're interseted in EDA, checkout this excellent notebook: 
https://www.kaggle.com/code/odins0n/playground-s-3-e-3-eda-modelling

#### Also, taking from "inspirations" from the following notebook, make sure you checkout this one as well:
https://www.kaggle.com/code/radek1/eda-training-a-1st-model-submission

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from pathlib import Path
import xgboost as xgb
import lightgbm as lgbm
import catboost
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from IPython.display import display
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
import optuna
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
BASE_PATH = Path('../input/playground-series-s3e3')

# id is not going to be an informative feature, so we're dropping it for train
# but since we'll need test set's ids to make the submission file, so we'll save those in  a separate varible before dropping
train = pd.read_csv(BASE_PATH / "train.csv").drop(columns="id")
test = pd.read_csv(BASE_PATH / "test.csv")
test_idx = test.id
test = test.drop(columns="id")

# It's been shown that incorporating original data, improves scores - at least on the public leaderboard. So let's do that!
original = pd.read_csv('../input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv')

train.head()

In [ ]:
original.head()

## Let's make the feature names and order consistent b/w our competition dataset and original dataset, before we concatenate

In [ ]:
original['Attrition'] = (original['Attrition'] == 'Yes').astype(np.int64)

# in original data, id is termed as "EmployeeNumber", so let's drop it
original.drop(columns="EmployeeNumber", inplace=True)

In [ ]:
# now reordering the features in original dataset
original = original[list(train.columns)]

In [ ]:
# let's finally concatenate
train_extended = pd.concat([train, original]).reset_index(drop=True)
len(train_extended)

In [ ]:
# checking for null values
pd.concat([train_extended.isnull().sum().rename("Missing in Train"),
           test.isnull().sum().rename("Missing in Test")], axis=1).sort_values(by="Missing in Train")

In [ ]:
## Insights: No missing values! Something to celebrate! :p

# Before we preprocess, let's concatenate test data to train

In [ ]:
y = train_extended.Attrition
df = pd.concat([train_extended.drop(columns="Attrition"), test])

## let's check for features types and see which we need to encode

In [ ]:
df.dtypes.sort_values()

## Remember, being of type int, doesn't mean that the feature cannot be categorial.
### So, let's check for unique values in each column

In [ ]:
df.nunique().sort_values()

### Taking a quick look at number of unique values in features reveals that we should be safe setting the threshold for to 20 unique values for what consitutes as a categorical feature
### We'll drop columns with only one value as they bring nothing to the table

### But feel free to use your own intution and test & trial to figure our what's works best in terms of threshold and features

In [ ]:
feats_to_drop = [col for col in df.columns if df[col].nunique()==1]
cat_features = [col for col in df.columns if df[col].nunique() <= 20 and df[col].nunique() > 1]

In [ ]:
df.drop(columns=feats_to_drop, inplace=True)

### We won't use one hot encoder here, because we already have a large ratio of features to rows and one hotting would increase that ratio by a large margin even further which will result in severe overfitting
### Rather we'll use ordinal/label encoder (they're basically the same thing)

In [ ]:
ord_enc = OrdinalEncoder()

ord_enc.fit(df[cat_features])

df[cat_features] = ord_enc.transform(df[cat_features])
df.head()

# Add some risk factors

In [ ]:
df['MonthlyIncome/Age'] = df['MonthlyIncome'] / df['Age']

df["Age_risk"] = (df["Age"] < 34).astype(int)
df["HourlyRate_risk"] = (df["HourlyRate"] < 60).astype(int)
df["Distance_risk"] = (df["DistanceFromHome"] >= 20).astype(int)
df["YearsAtCo_risk"] = (df["YearsAtCompany"] < 4).astype(int)

df['NumCompaniesWorked'] = df['NumCompaniesWorked'].replace(0, 1)
df['AverageTenure'] = df["TotalWorkingYears"] / df["NumCompaniesWorked"]
# df['YearsAboveAvgTenure'] = df['YearsAtCompany'] - df['AverageTenure']

df['JobHopper'] = ((df["NumCompaniesWorked"] > 2) & (df["AverageTenure"] < 2.0)).astype(int)

df["AttritionRisk"] = df["Age_risk"] + df["HourlyRate_risk"] + df["Distance_risk"] + df["YearsAtCo_risk"] + df['JobHopper']

# More feature engineering ideas
# https://www.kaggle.com/competitions/playground-series-s3e3/discussion/379093
df['feature_1'] = np.where(((df['StockOptionLevel'] >= 1) & 
                            (df['YearsAtCompany'] >= 3) & 
                            (df['YearsWithCurrManager'] >= 1)), 1, 0)
df['feature_2'] = np.where(((df['StockOptionLevel'] < 1) & 
                            (df['MonthlyIncome'] > 2700) & 
                            (df['OverTime'] == 'Yes')), 1, 0)

## Always a good idea to scale the features

Should we scale everything together, or separate them first?

In [ ]:
sc = StandardScaler()
df = sc.fit_transform(df)

### Let's seprate test and train sets

In [ ]:
X_train = df[:-len(test), :]
X_test = df[-len(test): , :]

In [ ]:
X_train

In [ ]:
y

# Modelling

### But first, let's setup cross validation

In [ ]:
def cross_validate(X, y, model):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1337)
    all_scores = []
    
    for fold_id, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict_proba(X_val)[:, 1]
        
        auc = roc_auc_score(y_val, y_pred)
        
        print(f"Fold {fold_id} \t auc: {auc}")
        
        all_scores.append(auc)
    
    avg_auc = np.mean(all_scores)
    
    print(f"Avg AUC: {avg_auc}")

## XGBoost

In [ ]:
# random params values - make sure to tune yours
xgb_params = {'n_estimators': 150,
                 'max_depth': 3,
                 'learning_rate': 0.1,
                 'min_child_weight': 4,
                 'subsample': 0.7,
                 'colsample_bytree': 0.3
             }


xgb_clf = xgb.XGBClassifier(**xgb_params)

cross_validate(X_train, y, xgb_clf)

xgb_clf.fit(X_train, y, verbose=0)

## LGBM Classifier

In [ ]:
# random params but feel free to tune
lgbm_params = {'n_estimators': 407,
                 'num_rounds': 274,
                 'learning_rate': 0.1,
                 'num_leaves': 195,
                 'max_depth': 9,
                 'min_data_in_leaf': 46,
                 'lambda_l1': 0.01,
                 'lambda_l2': 0.6,
                 'min_gain_to_split': 1.42,
                 'bagging_fraction': 0.45,
                 'feature_fraction': 0.3}

In [ ]:
lgbm_clf = lgbm.LGBMClassifier(**lgbm_params)

cross_validate(X_train, y, lgbm_clf)

lgbm_clf.fit(X_train, y, verbose=False)

## CatBoost

In [ ]:
#random params but feel free to tune
catboost_params = {'loss_function': 'CrossEntropy',
                     'learning_rate': 0.76,
                     'l2_leaf_reg': 0.014,
                     'colsample_bylevel': 0.06,
                     'depth': 1,
                     'boosting_type': 'Plain',
                     'bootstrap_type': 'Bernoulli',
                     'min_data_in_leaf': 18,
                     'one_hot_max_size': 14,
                     'subsample': 0.99}

catboost_clf = catboost.CatBoostClassifier(**catboost_params)

cross_validate(X_train, y, catboost_clf)

catboost_clf.fit(X_train, y, verbose=False)

In [ ]:
xgb_preds = xgb_clf.predict_proba(X_test)[:, 1]
lgbm_preds = lgbm_clf.predict_proba(X_test)[:, 1]
cat_preds = catboost_clf.predict_proba(X_test)[:, 1]

In [ ]:
# Give more weight to XGB
final_preds = np.column_stack([xgb_preds, xgb_preds,
                               # lgbm_preds,
                               cat_preds]).mean(axis=1)

In [ ]:
submission = pd.DataFrame({"id": test_idx, "Attrition": final_preds})
submission.head()

In [ ]:
submission['Attrition'].plot.hist();

In [ ]:
submission.to_csv("submission.csv", index=False)